        # Fake News Detection EDA

        This notebook performs exploratory data analysis on the LIAR dataset across the `train`, `valid`, and `test` splits.

        **Outputs**
        - Publication-quality figures saved to `notebooks/figures/`
        - All plots generated at 300 DPI with consistent labeling
        


In [ ]:
        from __future__ import annotations

        import sys
        from collections import Counter
        from pathlib import Path

        import matplotlib.pyplot as plt
        import numpy as np
        import pandas as pd
        import seaborn as sns
        from IPython.display import display
        from wordcloud import WordCloud

        CURRENT_DIR = Path.cwd().resolve()
        if (CURRENT_DIR / "backend").exists():
            PROJECT_ROOT = CURRENT_DIR
            NOTEBOOK_DIR = CURRENT_DIR / "notebooks"
        else:
            NOTEBOOK_DIR = CURRENT_DIR
            PROJECT_ROOT = CURRENT_DIR.parent

        BACKEND_DIR = PROJECT_ROOT / "backend"
        FIGURES_DIR = NOTEBOOK_DIR / "figures"
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)

        if str(BACKEND_DIR) not in sys.path:
            sys.path.insert(0, str(BACKEND_DIR))

        from data_utils import LIAR_COLUMNS, preprocess_liar_dataframe, resolve_data_directory
        from features import calculate_credibility_from_counts, tokenize_statement

        sns.set_theme(style="whitegrid", context="talk")
        plt.rcParams.update(
            {
                "figure.dpi": 120,
                "savefig.dpi": 300,
                "figure.facecolor": "white",
                "axes.facecolor": "white",
                "axes.titlesize": 18,
                "axes.labelsize": 14,
                "legend.fontsize": 11,
                "xtick.labelsize": 11,
                "ytick.labelsize": 11,
            }
        )

        ORIGINAL_LABEL_ORDER = [
            "pants-fire",
            "false",
            "barely-true",
            "half-true",
            "mostly-true",
            "true",
        ]
        BINARY_LABEL_NAMES = {0: "FAKE", 1: "REAL"}
        


In [ ]:
        def save_figure(filename: str) -> None:
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches="tight")
            plt.show()


        def load_split_with_original_label(split_name: str, data_dir: Path) -> pd.DataFrame:
            split_path = data_dir / f"{split_name}.tsv"
            raw_frame = pd.read_csv(split_path, sep="\t", header=None, names=LIAR_COLUMNS)
            raw_frame["original_label"] = raw_frame["label"]
            processed_frame = preprocess_liar_dataframe(raw_frame)
            processed_frame["binary_label_name"] = processed_frame["label"].map(BINARY_LABEL_NAMES)
            processed_frame["statement_length"] = processed_frame["statement"].str.len()
            processed_frame["speaker_credibility"] = processed_frame.apply(calculate_credibility_from_counts, axis=1)
            return processed_frame
        


In [ ]:
        data_dir = resolve_data_directory(PROJECT_ROOT / "data" if (PROJECT_ROOT / "data").exists() else None)

        splits = {
            split_name: load_split_with_original_label(split_name, data_dir)
            for split_name in ("train", "valid", "test")
        }

        train_df = splits["train"]
        valid_df = splits["valid"]
        test_df = splits["test"]
        full_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)

        pd.DataFrame(
            {
                "split": ["train", "valid", "test", "all"],
                "rows": [len(train_df), len(valid_df), len(test_df), len(full_df)],
            }
        )
        


        ## Label Distribution

        The first figure shows the original six-way LIAR labels, followed by the binary `FAKE` vs `REAL` mapping used by the backend.
        


In [ ]:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        original_counts = (
            full_df["original_label"]
            .value_counts()
            .reindex(ORIGINAL_LABEL_ORDER)
            .fillna(0)
            .astype(int)
        )
        sns.barplot(
            x=original_counts.index,
            y=original_counts.values,
            palette="magma",
            ax=axes[0],
        )
        axes[0].set_title("LIAR Label Distribution (6 Classes)")
        axes[0].set_xlabel("Original label")
        axes[0].set_ylabel("Number of statements")
        axes[0].tick_params(axis="x", rotation=30)

        binary_counts = (
            full_df["binary_label_name"]
            .value_counts()
            .reindex(["FAKE", "REAL"])
            .fillna(0)
            .astype(int)
        )
        sns.barplot(
            x=binary_counts.index,
            y=binary_counts.values,
            palette=["#d7301f", "#1a9850"],
            ax=axes[1],
        )
        axes[1].set_title("Binary Label Distribution")
        axes[1].set_xlabel("Binary label")
        axes[1].set_ylabel("Number of statements")

        save_figure("eda_label_distribution.png")
        


        ## Top 20 Most Frequent Words Per Class

        Each class gets a two-panel figure: a word cloud and a ranked frequency bar chart based on the backend tokenizer.
        


In [ ]:
        class_frequency_rows = []

        for label_name in ORIGINAL_LABEL_ORDER:
            statements = full_df.loc[full_df["original_label"] == label_name, "statement"]
            token_counts = Counter()
            for statement in statements:
                token_counts.update(tokenize_statement(statement))

            top_20 = token_counts.most_common(20)
            class_frequency_rows.extend(
                {"label": label_name, "token": token, "count": count}
                for token, count in top_20
            )

            if not top_20:
                continue

            fig, axes = plt.subplots(1, 2, figsize=(18, 7))
            cloud = WordCloud(
                width=1200,
                height=800,
                background_color="white",
                colormap="inferno",
                max_words=100,
            ).generate_from_frequencies(dict(top_20))

            axes[0].imshow(cloud, interpolation="bilinear")
            axes[0].axis("off")
            axes[0].set_title(f"Word Cloud: {label_name}")

            words, counts = zip(*top_20)
            sns.barplot(
                x=list(counts),
                y=list(words),
                palette="rocket",
                ax=axes[1],
            )
            axes[1].set_title(f"Top 20 Tokens: {label_name}")
            axes[1].set_xlabel("Frequency")
            axes[1].set_ylabel("Token")

            save_figure(f"eda_top_words_{label_name.replace('-', '_')}.png")

        frequency_summary = pd.DataFrame(class_frequency_rows)
        display(frequency_summary.head(20))
        


        ## Statement Length Distribution

        This plot highlights the distribution of statement lengths, separated by the binary target used in deployment.
        


In [ ]:
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.histplot(
            data=full_df,
            x="statement_length",
            hue="binary_label_name",
            bins=40,
            kde=True,
            palette={"FAKE": "#d7301f", "REAL": "#1a9850"},
            ax=ax,
        )
        ax.set_title("Statement Length Distribution")
        ax.set_xlabel("Statement length (characters)")
        ax.set_ylabel("Count")

        save_figure("eda_statement_length_distribution.png")
        


        ## Speaker Credibility Analysis

        The scatter plot compares each speaker's false-rate estimate against the number of statements attributed to them in the dataset.
        


In [ ]:
        speaker_summary = (
            full_df.groupby("speaker")
            .agg(
                dataset_statement_count=("id", "count"),
                false_counts=("false_counts", "mean"),
                pants_on_fire_counts=("pants_on_fire_counts", "mean"),
                barely_true_counts=("barely_true_counts", "mean"),
                half_true_counts=("half_true_counts", "mean"),
                mostly_true_counts=("mostly_true_counts", "mean"),
            )
            .reset_index()
        )

        speaker_summary["credibility_total"] = (
            speaker_summary["false_counts"]
            + speaker_summary["pants_on_fire_counts"]
            + speaker_summary["barely_true_counts"]
            + speaker_summary["half_true_counts"]
            + speaker_summary["mostly_true_counts"]
        )
        speaker_summary["false_rate"] = (
            speaker_summary["false_counts"] + speaker_summary["pants_on_fire_counts"]
        ) / speaker_summary["credibility_total"].replace(0, np.nan)
        speaker_summary["false_rate"] = speaker_summary["false_rate"].fillna(0)

        fig, ax = plt.subplots(figsize=(12, 7))
        sns.scatterplot(
            data=speaker_summary,
            x="dataset_statement_count",
            y="false_rate",
            size="credibility_total",
            sizes=(20, 350),
            alpha=0.7,
            color="#7b3294",
            ax=ax,
        )
        ax.set_title("Speaker Credibility Analysis")
        ax.set_xlabel("Number of statements in dataset")
        ax.set_ylabel("Estimated false rate")

        save_figure("eda_speaker_credibility_scatter.png")
        


        ## Party Distribution

        Party categories are aggregated into a pie chart to reveal how political affiliation is represented across the dataset.
        


In [ ]:
        party_counts = full_df["party"].value_counts()
        top_parties = party_counts.head(7).copy()
        if len(party_counts) > 7:
            top_parties.loc["other"] = party_counts.iloc[7:].sum()

        fig, ax = plt.subplots(figsize=(10, 10))
        ax.pie(
            top_parties.values,
            labels=top_parties.index,
            autopct="%1.1f%%",
            startangle=90,
            textprops={"fontsize": 11},
        )
        ax.set_title("Party Distribution")

        save_figure("eda_party_distribution_pie.png")
        


        ## Correlation Heatmap of Numeric Credibility Features

        This heatmap focuses on the credibility-related count features and derived speaker credibility score.
        


In [ ]:
        correlation_columns = [
            "barely_true_counts",
            "false_counts",
            "half_true_counts",
            "mostly_true_counts",
            "pants_on_fire_counts",
            "speaker_credibility",
            "statement_length",
        ]

        correlation_matrix = full_df[correlation_columns].corr(numeric_only=True)

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            correlation_matrix,
            annot=True,
            fmt=".2f",
            cmap="mako",
            square=True,
            linewidths=0.5,
            cbar_kws={"label": "Correlation"},
            ax=ax,
        )
        ax.set_title("Correlation Heatmap of Numeric Credibility Features")

        save_figure("eda_correlation_heatmap.png")
        
